In [2]:
import pandas as pd
fixed_entries = [ {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.", "keywords": "fee cost price charge", "category": "billing"},
                  {"question": "how to reset password", "answer": "Go to Settings > Reset Password.", "keywords": "password reset login", "category": "account"},
                  {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.", "keywords": "hours timing open time", "category": "general"},
                  {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.", "keywords": "pay payment upi fee", "category": "billing"}, ]
#my roll no is 1024170448 so for 4%3=1=account 8%3=2=general
my_entries = [
    {
        "question": "how do i update my registered mobile number",
        "answer": "Go to Profile > Edit Details to update your phone number.",
        "keywords": "mobile number update phone account contact details",
        "category": "account"
    },
    {
        "question": "Are meals included",
        "answer": "Yes, lunch is provided",
        "keywords": "meal lunch refreshments food",
        "category": "general"
    }
]
df=pd.DataFrame(fixed_entries+my_entries)
print(df)

                                      question  \
0                       what is the annual fee   
1                        how to reset password   
2                  what are your working hours   
3                        how can i pay the fee   
4  how do i update my registered mobile number   
5                           Are meals included   

                                              answer  \
0                          The annual fee is Rs 500.   
1                   Go to Settings > Reset Password.   
2                          We are open 9 AM to 5 PM.   
3         You can pay via UPI, card, or net banking.   
4  Go to Profile > Edit Details to update your ph...   
5                             Yes, lunch is provided   

                                            keywords category  
0                              fee cost price charge  billing  
1                               password reset login  account  
2                             hours timing open time  general  


In [11]:
def score_query(query, df):
    # Split user's query into individual lowercase words
    query_words = set(query.lower().split())

    scores = []

    # Loop through each row in the DataFrame
    for index, row in df.iterrows():
        # Combine the row's question and keywords
        combined_text = (row['question'] + " " + row['keywords']).lower()
        row_words = set(combined_text.split())

        # Count how many words match between user query and this row
        matching_words = query_words.intersection(row_words)

        # Simple score = count of matching words
        score = len(matching_words)
        scores.append(score)

    # Add the scores as a new column
    df_copy = df.copy()
    df_copy['confidence_score'] = scores

    # Keep only rows where score > 0, and sort from highest to lowest score
    results = df_copy[df_copy['confidence_score'] > 0]
    results = results.sort_values(by='confidence_score', ascending=False)


    return results[['question', 'answer', 'category', 'confidence_score']]

In [16]:

user_input = "how can i pay annual fee"
ans2 = score_query(user_input, df)
print(ans2.to_string(index=False))

 Single best match found with score  5 :
             question                                     answer category  confidence_score
how can i pay the fee You can pay via UPI, card, or net banking.  billing                 5

--------------------------------------------------
                                   question                                                    answer category  confidence_score
                      how can i pay the fee                You can pay via UPI, card, or net banking.  billing                 5
                     what is the annual fee                                 The annual fee is Rs 500.  billing                 2
how do i update my registered mobile number Go to Profile > Edit Details to update your phone number.  account                 2
                      how to reset password                          Go to Settings > Reset Password.  account                 1


In [7]:
def same_category(category_name, df):
    # Filter DataFrame where category matches the input category_name
    return df[df['category'] == category_name.lower()]
q3 = same_category("account", df)
print(q3[['question', 'answer', 'category']].to_string(index=False))

                                   question                                                    answer category
                      how to reset password                          Go to Settings > Reset Password.  account
how do i update my registered mobile number Go to Profile > Edit Details to update your phone number.  account


In [9]:
print(df.loc[5,['question','keywords']])
inp=input("Enter new keyword")
df.loc[5,'keywords']+=inp
print("updated keywords")
print(df.loc[5,['question','keywords']].to_string(index=False))
df.to_csv("1024170448_faq_data.csv",index=False)


question                    Are meals included
keywords    meal lunch refreshments foodkhanaa
Name: 5, dtype: object
updated keywords
                       Are meals included
meal lunch refreshments foodkhanaa khanaa


In [10]:
# Using count() on a specific column
category_counts = df.groupby("category")["question"].count()
print(category_counts)

category
account    2
billing    2
general    2
Name: question, dtype: int64


In [13]:
def score_query_with_ties(query, df):
    # Split user's query into individual lowercase words
    query_words = set(query.lower().split())

    scores = []

    # Loop through each row in the DataFrame
    for index, row in df.iterrows():
        # Combine the row's question and keywords
        combined_text = (row['question'] + " " + row['keywords']).lower()
        row_words = set(combined_text.split())

        # Count how many words match between user query and this row
        matching_words = query_words.intersection(row_words)

        # Simple score = count of matching words
        score = len(matching_words)
        scores.append(score)

    # Add the scores as a new column
    df_copy = df.copy()
    df_copy['confidence_score'] = scores

    # Keep only rows where score > 0, and sort from highest to lowest score
    results = df_copy[df_copy['confidence_score'] > 0]
    results = results.sort_values(by='confidence_score', ascending=False)
    #to handle questions with same confidence score
    if not results.empty:
        # Find the highest score achieved
        max_score = results['confidence_score'].max()

        # Get all entries tied for the highest score
        top_matches = results[results['confidence_score'] == max_score]

        # Check if there is a tie
        if len(top_matches) > 1:
            print("TIE DETECTED: Found ",len(top_matches)," equally good matches with score ",max_score,":")
        else:
            print(" Single best match found with score ",max_score,":")

        print(top_matches[['question', 'answer', 'category', 'confidence_score']].to_string(index=False))
        print("\n" + "-"*50)
    else:
        print("No matching FAQs found.")

    return results[['question', 'answer', 'category', 'confidence_score']]

In [14]:
score_query_with_ties("fee", df)

TIE DETECTED: Found  2  equally good matches with score  1 :
              question                                     answer category  confidence_score
what is the annual fee                  The annual fee is Rs 500.  billing                 1
 how can i pay the fee You can pay via UPI, card, or net banking.  billing                 1

--------------------------------------------------


,question,answer,category,confidence_score
0,what is the annual fee,The annual fee is Rs 500.,billing,1
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",billing,1


In [15]:
score_query_with_ties("reset password", df)

 Single best match found with score  2 :
             question                           answer category  confidence_score
how to reset password Go to Settings > Reset Password.  account                 2

--------------------------------------------------


,question,answer,category,confidence_score
1,how to reset password,Go to Settings > Reset Password.,account,2
